Tên: Phạm Văn Thắng

## Bài 1: Titanic Dataset

### Yêu cầu

1. Sử dụng **Logistic Regression** để dự đoán hành khách sống sót hay không.
2. Sử dụng cùng cách chia dữ liệu train/test như bài tập trước nếu có thể.
3. Đánh giá kết quả của mô hình Logistic Regression.
4. So sánh kết quả của **Logistic Regression** với kết quả của **Linear Regression** trong bài tập trước.
5. Đưa ra nhận xét về mô hình phù hợp hơn đối với bài toán phân loại hành khách sống sót.

### 1. Tải và làm sạch dữ liệu:

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Sẵn sàng.")

Sẵn sàng.


In [41]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")

# Bỏ các cột rò rỉ nhãn / dư thừa (giống Task 1 - Buổi 5)
leaky = ["alive", "who", "adult_male", "class", "deck", "embark_town", "alone"]
leaky = [c for c in leaky if c in df.columns]
df = df.drop(columns=leaky)

print("Các cột còn lại:", list(df.columns))
df.head()

Đã tải từ seaborn.
Các cột còn lại: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


### 2. Chia train/validation/test

In [42]:
X = df.drop(columns="survived")
y = df["survived"]

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=42, stratify=y_tmp
)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  tỷ lệ survived ({name}): {yy.mean():.3f}")

Train: (623, 7) | Val: (134, 7) | Test: (134, 7)
  tỷ lệ survived (train): 0.384
  tỷ lệ survived (val): 0.388
  tỷ lệ survived (test): 0.381


### 3. Pipeline

In [43]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train) 
X_train_t = preprocess.transform(X_train)
X_val_t   = preprocess.transform(X_val)
X_test_t  = preprocess.transform(X_test)

feature_names = list(preprocess.get_feature_names_out())
print(X_train_t.shape, X_val_t.shape, X_test_t.shape)
print(feature_names)

(623, 8) (134, 8) (134, 8)
['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_male', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass']


### 4. Logistic Regression

In [44]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_t, y_train)

y_val_pred_log = log_reg.predict(X_val_t)
y_test_pred_log = log_reg.predict(X_test_t)

print("=== Logistic Regression — Validation ===")
print(classification_report(y_val, y_val_pred_log, target_names=["Không sống sót", "Sống sót"]))

=== Logistic Regression — Validation ===
                precision    recall  f1-score   support

Không sống sót       0.88      0.82      0.85        82
      Sống sót       0.74      0.83      0.78        52

      accuracy                           0.82       134
     macro avg       0.81      0.82      0.81       134
  weighted avg       0.83      0.82      0.82       134



### 5. Linear Regression

In [45]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_t, y_train)

y_val_pred_lin_raw = lin_reg.predict(X_val_t)
y_test_pred_lin_raw = lin_reg.predict(X_test_t)

y_val_pred_lin = (y_val_pred_lin_raw >= 0.5).astype(int)
y_test_pred_lin = (y_test_pred_lin_raw >= 0.5).astype(int)

print("Ví dụ giá trị dự đoán thô của Linear Regression (chưa threshold):")
print(np.round(y_val_pred_lin_raw[:10], 3))
print("\n=== Linear Regression (threshold 0.5) — Validation ===")
print(classification_report(y_val, y_val_pred_lin, target_names=["Không sống sót", "Sống sót"]))

Ví dụ giá trị dự đoán thô của Linear Regression (chưa threshold):
[0.148 0.03  0.461 1.029 0.151 0.132 0.089 0.208 0.987 0.327]

=== Linear Regression (threshold 0.5) — Validation ===
                precision    recall  f1-score   support

Không sống sót       0.88      0.85      0.86        82
      Sống sót       0.78      0.81      0.79        52

      accuracy                           0.84       134
     macro avg       0.83      0.83      0.83       134
  weighted avg       0.84      0.84      0.84       134



### 6. So sánh

In [46]:
def get_metrics(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
    }

results = pd.DataFrame([
    get_metrics(y_test, y_test_pred_log, "Logistic Regression"),
    get_metrics(y_test, y_test_pred_lin, "Linear Regression (threshold=0.5)"),
]).set_index("Model").round(4)

results

,Accuracy,Precision,Recall,F1-score
Model,,,,
Logistic Regression,0.7836,0.7500,0.6471,0.6947
Linear Regression (threshold=0.5),0.7612,0.7021,0.6471,0.6735


### 7. Nhận xét:

**So sánh kết quả:**

- **Logistic Regression** thường cho Accuracy/F1 tương đương hoặc nhỉnh hơn Linear Regression.
- **Linear Regression** không được thiết kế cho phân loại: đầu ra có thể **âm hoặc lớn hơn 1**, không phải xác suất thật sự.
  Việc threshold ở 0.5 chỉ là một cách để ép về bài toán nhị phân.
- Vì vậy, với bài toán phân loại "sống sót hay không", **Logistic Regression là lựa chọn phù hợp hơn** cả về mặt lý thuyết
  (hàm mất mát log-loss phù hợp với nhãn nhị phân, đầu ra là xác suất) lẫn thực nghiệm (chỉ số đánh giá phân loại ổn định
  và đáng tin cậy hơn).

## Bài 2: Dry Bean Dataset

Xây dựng mô hình phân loại các loại hạt trong bộ dữ liệu **Dry Bean Dataset** bằng hai thuật toán:

1. **Logistic Regression**
2. **K-Nearest Neighbors – KNN**


In [47]:
train_df = pd.read_csv("D:\HCMUT\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_train.csv")
test_df  = pd.read_csv("D:\HCMUT\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head()

Train shape: (10834, 17)
Test shape : (2709, 17)


<>:1: SyntaxWarning: invalid escape sequence '\H'
<>:2: SyntaxWarning: invalid escape sequence '\H'
<>:1: SyntaxWarning: invalid escape sequence '\H'
<>:2: SyntaxWarning: invalid escape sequence '\H'
C:\Users\Admin\AppData\Local\Temp\ipykernel_12012\1562086129.py:1: SyntaxWarning: invalid escape sequence '\H'
  train_df = pd.read_csv("D:\HCMUT\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_train.csv")
C:\Users\Admin\AppData\Local\Temp\ipykernel_12012\1562086129.py:2: SyntaxWarning: invalid escape sequence '\H'
  test_df  = pd.read_csv("D:\HCMUT\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_test.csv")


,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,69471,1069.638,399.100245,225.005782,1.773733,0.825923,71088,297.410868,0.707386,0.977254,0.763027,0.745203,0.005745,0.001093,0.555328,0.985004,CALI
1,82877,1162.581,391.817013,270.836144,1.446694,0.722634,84171,324.841921,0.825986,0.984627,0.770544,0.829065,0.004728,0.001378,0.687349,0.994384,BARBUNYA
2,65042,1023.506,419.202858,198.962774,2.106941,0.880190,65748,287.774298,0.783403,0.989262,0.780231,0.686480,0.006445,0.000883,0.471255,0.992906,HOROZ
3,41315,758.920,287.438268,183.447580,1.566869,0.769858,41704,229.355383,0.791930,0.990672,0.901417,0.797929,0.006957,0.001740,0.636691,0.997611,SIRA
4,91088,1168.645,459.300729,253.950486,1.808623,0.833243,91799,340.553731,0.789051,0.992255,0.838119,0.741461,0.005042,0.000940,0.549765,0.994318,CALI


In [48]:
target = "class"
features = [c for c in train_df.columns if c != target]

X_train, y_train = train_df[features].copy(), train_df[target].copy()
X_test,  y_test  = test_df[features].copy(),  test_df[target].copy()

print("Số feature:", len(features))
print("Các lớp:", sorted(y_train.unique()))
print("\nPhân bố lớp trong train:")
print(y_train.value_counts().sort_index())

Số feature: 16
Các lớp: ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']

Phân bố lớp trong train:
class
BARBUNYA    1057
BOMBAY       418
CALI        1304
DERMASON    2837
HOROZ       1488
SEKER       1621
SIRA        2109
Name: count, dtype: int64


### 1. Chuẩn hóa dữ liệu

In [49]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

class_names = le.classes_
print("Các lớp (đã encode):", dict(zip(class_names, range(len(class_names)))))

Các lớp (đã encode): {'BARBUNYA': 0, 'BOMBAY': 1, 'CALI': 2, 'DERMASON': 3, 'HOROZ': 4, 'SEKER': 5, 'SIRA': 6}


### 2. Mô hình 1: Logistic Regression

In [50]:
log_reg = LogisticRegression(max_iter=2000, random_state=42)
log_reg.fit(X_train_scaled, y_train_enc)

y_pred_log = log_reg.predict(X_test_scaled)

print("=== Logistic Regression — Test ===")
print(classification_report(y_test_enc, y_pred_log, target_names=class_names))

=== Logistic Regression — Test ===
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709



### 3. Mô hình 2: KNN

In [51]:
param_grid = {"n_neighbors": [3, 5, 7, 9, 11, 15]}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
)
grid_knn.fit(X_train_scaled, y_train_enc)

print("k tốt nhất:", grid_knn.best_params_)
print("F1-macro (CV) tốt nhất:", round(grid_knn.best_score_, 4))

knn = grid_knn.best_estimator_
y_pred_knn = knn.predict(X_test_scaled)

print("\n=== KNN — Test ===")
print(classification_report(y_test_enc, y_pred_knn, target_names=class_names))

k tốt nhất: {'n_neighbors': 15}
F1-macro (CV) tốt nhất: 0.9371

=== KNN — Test ===
              precision    recall  f1-score   support

    BARBUNYA       0.96      0.88      0.92       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.90      0.96      0.93       326
    DERMASON       0.91      0.90      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.85      0.88      0.86       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709



### 4. So sánh

In [52]:
def get_metrics(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro"),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro"),
        "F1-score (macro)": f1_score(y_true, y_pred, average="macro"),
    }

results = pd.DataFrame([
    get_metrics(y_test_enc, y_pred_log, "Logistic Regression"),
    get_metrics(y_test_enc, y_pred_knn, f"KNN (k={grid_knn.best_params_['n_neighbors']})"),
]).set_index("Model").round(4)

results

,Accuracy,Precision (macro),Recall (macro),F1-score (macro)
Model,,,,
Logistic Regression,0.9192,0.9307,0.9300,0.9302
KNN (k=15),0.9158,0.9312,0.9276,0.9290


### 5. Nhận xét:

- Cả hai mô hình đều đạt độ chính xác khá cao trên bộ Dry Bean vì các đặc trưng hình học (area, perimeter,
  shape factors...) khá tách biệt giữa các loại hạt, đặc biệt lớp **BOMBAY** (hạt lớn khác biệt hẳn) thường được
  phân loại gần như hoàn hảo bởi cả hai mô hình.
- **Logistic Regression** huấn luyện nhanh, mô hình tuyến tính, dễ diễn giải hệ số, nhưng có thể gặp khó khi ranh
  giới giữa các lớp không tuyến tính.
- **KNN** có thể nắm bắt ranh giới quyết định phức tạp/phi tuyến tốt hơn vì dựa trực tiếp vào khoảng cách tới các
  điểm lân cận, nhưng chi phí dự đoán (tính khoảng cách tới toàn bộ tập train) tốn kém hơn khi dữ liệu lớn, và
  **rất nhạy với việc scaling** — đây là lý do bắt buộc phải `StandardScaler` trước khi huấn luyện.